In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
df = pd.read_csv('data.csv', index_col='Date', parse_dates=True)
data_raw = df[['Open', 'High', 'Low', 'Volume', 'Close']].values

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data_raw)

In [ ]:
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length), :-1] # Open, High, Low, Volume
        y = data[i + seq_length, -1]      # Target: Close
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

WINDOW_SIZE = 10 # Окно в 10 торговых дней
X, y = create_sequences(data_scaled, WINDOW_SIZE)

train_size = int(len(X) * 0.8)
X_train, X_test = torch.FloatTensor(X[:train_size]), torch.FloatTensor(X[train_size:])
y_train, y_test = torch.FloatTensor(y[:train_size]), torch.FloatTensor(y[train_size:])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=False)

In [ ]:
class StockRegressor(nn.Module):
    def __init__(self, input_dim, seq_len):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(), # (10, 4) to 40
            nn.Linear(input_dim * seq_len, 64),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1) # output
        )
        
    def forward(self, x):
        return self.net(x)

model = StockRegressor(input_dim=4, seq_len=WINDOW_SIZE)
criterion = nn.MSELoss() # Среднеквадратичная ошибка
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
epochs = 30
history = {'train_loss': []}

In [ ]:
for epoch in range(epochs):
    model.train()
    batch_losses = []
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x).squeeze()
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
    
    avg_loss = np.mean(batch_losses)
    history['train_loss'].append(avg_loss)
    if (epoch+1) % 5 == 0:
        print(f"Эпоха [{epoch+1}/{epochs}], Loss: {avg_loss:.6f}")

In [ ]:
model.eval()
with torch.no_grad():
    preds_scaled = model(X_test).squeeze().numpy()

# (Создаем временный массив для инверсии трансформера)
dummy = np.zeros((len(preds_scaled), 5))
dummy[:, -1] = preds_scaled
preds_unscaled = scaler.inverse_transform(dummy)[:, -1]

dummy_y = np.zeros((len(y_test), 5))
dummy_y[:, -1] = y_test.numpy()
y_test_unscaled = scaler.inverse_transform(dummy_y)[:, -1]

In [ ]:
plt.figure(figsize=(18, 5))

# Loss
plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], color='blue', label='MSE Loss')
plt.title('Обучение: Динамика Loss')
plt.xlabel('Эпоха')
plt.ylabel('MSE')
plt.grid(True)

# prediction
plt.subplot(1, 3, 2)
plt.plot(y_test_unscaled, label='Реальная цена', color='black', alpha=0.6)
plt.plot(preds_unscaled, label='Предсказание', color='red', linestyle='--')
plt.title('Прогноз цен акций')
plt.legend()
plt.grid(True)

# True vs Pred
plt.subplot(1, 3, 3)
plt.scatter(y_test_unscaled, preds_unscaled, alpha=0.3, s=10)
plt.plot([y_test_unscaled.min(), y_test_unscaled.max()], 
         [y_test_unscaled.min(), y_test_unscaled.max()], 'r--')
plt.title('Диаграмма отклонений (True vs Pred)')
plt.xlabel('Реальная цена')
plt.ylabel('Предсказанная цена')
plt.grid(True)

plt.tight_layout()
plt.show()